# UNBLIND 00 — Provenance guards & data foundation

**Shared foundation for the DLA (`UNBLIND_01_*`) and sub-DLA (`UNBLIND_02_*`) unblinding
notebooks.** Run / import this first: it (1) refuses to proceed on an unstamped or
non-re-derivable artifact, (2) validates the headline JSON schema, (3) loads a tidy,
guarded data structure, (4) tabulates the carried systematics as data, and (5) self-checks
that no private value leaks into committed outputs.

> ## 🔴 PRIVACY — read before you run
> Real-LOA (DESI DR2 Loa) **result values** — dN/dX, Ω, f(N), per-z tables, over-count
> percentages — are **PRIVATE**. This notebook loads them into memory to *plot*, but must
> **never print, tabulate, or paraphrase** a single one. **Executed outputs contain private
> values → the committed notebook must have ZERO outputs.** Before committing:
> ```
> jupyter nbconvert --clear-output --inplace notebooks/UNBLIND_00_guards_and_data.ipynb
> ```
> (git sha stamps, array *shapes*, pass/fail, and *systematic magnitudes* are NOT results and
> are safe to show.)

All heavy lifting lives in the thin, testable package `CDDF_analysis/unblind/`; the cells
below drive and explain it.

## 0 — Environment & imports

Run with the project env and single-threaded BLAS:
```
conda activate gpdla
export OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1 HDF5_USE_FILE_LOCKING=FALSE
```

In [ ]:
import os, sys, json, subprocess, tempfile
import numpy as np

# Put the repo root on sys.path so `CDDF_analysis.unblind` imports regardless of CWD.
REPO = subprocess.run(["git", "rev-parse", "--show-toplevel"],
                      capture_output=True, text=True).stdout.strip()
if REPO and REPO not in sys.path:
    sys.path.insert(0, REPO)

from CDDF_analysis.unblind import provenance as prov
from CDDF_analysis.unblind import schema as schema
from CDDF_analysis.unblind import loader as loader
from CDDF_analysis.unblind import systematics as systematics
from CDDF_analysis.unblind import privacy as privacy

HEAD = prov._full_sha("HEAD", REPO)
print("repo :", REPO)
print("HEAD :", HEAD)

## 1 — The provenance guard (the point of this notebook)

A stamped headline is only worth anything if a third party can **re-derive it from
committed code**. This project has been burned repeatedly:

* a headline that lived only as **literals in a plot script** (no routine, no stamp);
* a stamped JSON whose **generating routine was never committed** and was later deleted;
* an artifact stamped with a **real, ancestor, non-dirty commit that nevertheless does not
  contain the routine** — `lls_recovery_figures.json` @ `78c01f6`, whose routine
  `lls_recovery_figures.py` was first committed later in `a907127`. Syntactically valid,
  semantically worthless.

So the guard classifies every artifact into a **4-way status** (plus refusal sub-states)
and only lets a notebook proceed on `RE_DERIVABLE`:

| status | meaning |
|---|---|
| `NOT_STAMPED` | `code_commit` missing or `"unknown"` |
| `DIRTY` | `code_commit` ends with `-dirty` (routine modified / tree unclean at gen time) |
| `ORPHANED` | commit exists but does **not contain** the generating routine |
| `RE_DERIVABLE` | commit exists, **contains the routine**, ancestor-of-or-equal to HEAD ✅ |
| *(refuse)* `COMMIT_NOT_FOUND` / `NO_ROUTINE` / `NOT_ANCESTOR` | other non-re-derivable cases |

The routine path is resolved from `metadata.routine`, else parsed from `metadata.rederive`,
else an explicit `routine_path=` argument. `RE_DERIVABLE` still **warns loudly** when HEAD has
moved past the stamp, or when the routine's blob changed since the stamp (drift).

The guard touches **only** `metadata.code_commit` + the routine path — never a science value.

### 1a — Demonstrate all four states (inline metadata → real git checks)

These use real commit SHAs from this repo's history, so the git checks are genuine.

In [ ]:
demo = {
    "NOT_STAMPED (unknown)": {"code_commit": "unknown"},
    "NOT_STAMPED (missing)": {},
    "DIRTY":  {"code_commit": HEAD + "-dirty",
               "rederive": "python CDDF_analysis/diagnostics/lls/break_census.py --force"},
    # ORPHANED: real commit 78c01f6 does NOT contain lls_recovery_figures.py (added in a907127).
    "ORPHANED (lls_recovery_figures @ 78c01f6)":
        {"code_commit": "78c01f650b5183a58f5b3d2cb032c5539a1af3a8",
         "rederive": "python CDDF_analysis/diagnostics/lls/lls_recovery_figures.py --force"},
    # RE_DERIVABLE: break_census @ a907127 (strict ancestor -> head_moved warning fires).
    "RE_DERIVABLE (break_census @ a907127)":
        {"code_commit": "a90712779d064ae4a3b354c4eeaaa2bb15ad1375",
         "rederive": "python CDDF_analysis/diagnostics/lls/break_census.py --force"},
}
print(f"{'case':46s} {'status':16s} head_moved drift")
for name, meta in demo.items():
    r = prov.classify(meta)
    print(f"{name:46s} {r.status:16s} {str(r.head_moved):>10s} {str(r.routine_drift)}")

### 1b — Demonstrate the guard on the two real headline files

`track_c_tf_loa_loa0.json` carries `code_commit="unknown"` and **must be rejected**.
The regenerated `..._restamped.json` (`d496f42`) **must be accepted**. We print pass/fail
only — **no science value from either file**. (These SCRATCH paths are per-user; the cell
degrades gracefully if they are absent.)

In [ ]:
loa0_dir = os.path.dirname(loader.DEFAULT_LOA0_ARTIFACT)
unknown_file  = os.path.join(loa0_dir, "track_c_tf_loa_loa0.json")   # code_commit="unknown"
restamp_file  = loader.DEFAULT_LOA0_ARTIFACT                          # code_commit=d496f42

if os.path.exists(unknown_file):
    try:
        prov.check_artifact(unknown_file, routine_path=loader.DEFAULT_LOA0_ROUTINE)
        print("REJECT-TEST: FAILED — unknown-stamped artifact was accepted (BUG)")
    except prov.ProvenanceError as e:
        print("REJECT-TEST: PASS — unknown-stamped artifact rejected:")
        print("   ", str(e)[:100])
else:
    print("REJECT-TEST: skipped (scratch file absent)")

if os.path.exists(restamp_file):
    r = prov.check_artifact(restamp_file, routine_path=loader.DEFAULT_LOA0_ROUTINE, verbose=False)
    print(f"ACCEPT-TEST: PASS — restamped artifact status = {r.status}"
          + ("  (HEAD moved past stamp)" if r.head_moved else "  (stamp == HEAD)"))
else:
    print("ACCEPT-TEST: skipped (scratch file absent)")

## 2 — Schema validator (fail loudly on drift)

The headline JSON (`track_c_tf_loa*` family) has a verified structure. The validator checks
**key presence and array shapes only** (never a value, so it is privacy-safe on the real file)
and raises `SchemaError` on any drift, so a plotter can never silently mis-read a re-shaped file:

* top-level `measurement`, `metadata`, `perz_fN`, `zbins`;
* `measurement['20.0'|'20.3']['dndx'|'omega']` → `integrated` (dict) + `perz` (5, one per z-bin),
  each with quantile keys `MAP,q025,q16,q84,q975,std`;
* `zbins` length 6 (→ 5 z-bins);
* `perz_fN`: `logN_centers`[52], `zbins`[6], `z_extrapolated`/`z_thin`/`truth_counts_perz`[5],
  and `perz`[5] each with `f/f68_lo/f68_hi/f95_lo/f95_hi`[52].

Extra keys (`perz_fN.band_method`, `perz_fN.floor`) are tolerated; only drift in required
keys/shapes fails.

In [ ]:
# Validate the loa0 headline (shapes only). Falls back to the purity artifact if needed.
target = loader.DEFAULT_LOA0_ARTIFACT if os.path.exists(loader.DEFAULT_LOA0_ARTIFACT) \
         else loader.DEFAULT_PURITY_ARTIFACT
with open(target) as f:
    d = json.load(f)
report = schema.validate_headline_schema(d)
print(report)

# Drift detection: drop one logN center on a copy -> must raise.
import copy
bad = copy.deepcopy(d); bad["perz_fN"]["logN_centers"] = bad["perz_fN"]["logN_centers"][:-1]
try:
    schema.validate_headline_schema(bad)
    print("\nDRIFT-TEST: FAILED — drift not caught (BUG)")
except schema.SchemaError as e:
    print("\nDRIFT-TEST: PASS — drift raised SchemaError:", e)

## 3 — Loader → tidy `HeadlineData` (+ three DISTINCT z-bin regime flags)

`load_headline()` runs the guard + schema validator, then returns numpy arrays for the
integrated scalars (with MC bands at both limits), the per-z dN/dX & Ω (with bands), and the
per-z f(N) (with 68/95 bands). **Paths are parameters with defaults**, not scattered literals.

### The z>4 extrapolation trap (why there are THREE booleans, not one)

z-bins: edges `[2.0, 2.5, 3.0, 3.5, 4.0, 4.25]`, centers `2.25 / 2.75 / 3.25 / 3.75 / 4.125`.
Two different ceilings **do not coincide**, and the loader refuses to hide it:

* `beyond_calibration` := stamped `metadata.z_extrapolated` — the frozen completeness
  `g(N,z)` has no 2LPT-0 truth support (lower edge above `max_truth_z ≈ 3.786`). → only the
  top bin `[4.0,4.25]`.
* `beyond_v2_fit` := `zbin_lo ≥ metadata.v2_z_fit_hi` (= 3.5) — the mean-flux / effective-
  optical-depth model was fit only up to 3.5; above it the **forward model is extrapolated**.
  → the `[3.5,4.0]` **and** `[4.0,4.25]` bins.
* `partial_truth_support` := `zbin_lo < max_truth_z < zbin_hi` — straddles the truth cap.
  → the `[3.5,4.0]` bin.

So the **z≈3.75 bin** (`[3.5,4.0]`) is *beyond the v2 fit ceiling* yet *not* flagged
`beyond_calibration` — a regime the stamped single flag misses. The loader emits a **loud
warning** for any such bin so a plotter cannot forget it.

In [ ]:
hd = loader.load_headline()   # defaults: loa0 restamped headline + its routine; guard + schema on
zf = hd.zflags

print("code_commit :", hd.code_commit, "  fp_estimator:", hd.fp_estimator,
      "  provenance:", hd.provenance.status)
print("limits      :", hd.limits)
print("z edges     :", hd.zbins.tolist())
print("z centers   :", hd.z_centers.tolist())
print("max_truth_z = %.4f   v2_z_fit_hi = %.2f" % (zf.max_truth_z, zf.v2_z_fit_hi))
print()
print(f"{'z-bin':14s} {'beyond_cal':>11s} {'beyond_v2':>10s} {'partial_truth':>14s} {'thin':>6s}")
for i in range(len(zf.z_centers)):
    lo, hi = zf.zbin_lo[i], zf.zbin_hi[i]
    print(f"[{lo:.2f},{hi:.2f}]   {str(zf.beyond_calibration[i]):>11s}"
          f" {str(zf.beyond_v2_fit[i]):>10s} {str(zf.partial_truth_support[i]):>14s}"
          f" {str(zf.z_thin[i]):>6s}")
print()
print("bins beyond v2 fit but NOT flagged beyond_calibration:", zf.v2_beyond_but_calibrated)

# Shapes only — NO science values.
print()
print("array shapes (values withheld — private):")
print("  integrated['20.3']['omega'] keys :", sorted(hd.integrated['20.3']['omega'].keys()))
print("  perz['20.3']['dndx']['MAP']      :", hd.perz['20.3']['dndx']['MAP'].shape)
print("  fN['f']                          :", hd.fN['f'].shape)
print("  fN['f68_lo'] / ['f95_hi']        :", hd.fN['f68_lo'].shape, hd.fN['f95_hi'].shape)
print("  logN_centers                     :", hd.logN_centers.shape)
print("  truth_counts_perz                :", hd.truth_counts_perz.shape, hd.truth_counts_perz.dtype)

## 4 — Carried systematics table (as data, VERIFIED / UNVERIFIED)

Downstream plots draw error bands / annotations from `systematics.carried_systematics()`.
Each row records **name, size, sign, what it applies to, the committed routine / stamped
artifact it came from, VERIFIED vs UNVERIFIED, and INSIDE vs OUTSIDE the plotted MC band.**

The plotted MC / independent band is **statistical only** (C/ρ calibration + real-sightline
bootstrap + NHI-measurement variance about a *frozen* calibration). Every calibration-transfer
/ model-choice / extrapolation term below is therefore **OUTSIDE** it and must be added by a
plotter — the band does not cover them.

The **sizes here are systematic magnitudes** (mostly mock/london-0-derived, several are
committed literals) — **not** real-LOA result values.

In [ ]:
rows = systematics.carried_systematics()
print(systematics.as_table(rows))
print()
for r in rows:
    print(f"* {r.name}  [{r.status} / band {r.band_relation}]")
    print(f"    size={r.size!r}  sign={r.sign!r}  applies_to={r.applies_to!r}")
    print(f"    source: {r.source}")
    if r.note:
        print(f"    note:   {r.note}")

**Row highlights**

* **deep-tail / mean-flux transfer ~12–13% on Ω(≥20.3)** — the DOMINANT term, VERIFIED
  (committed literal in `track_c_tf_loa.py` + london-0 closure). OUTSIDE the band.
* **BAL FP residual ~2–6%** — sub-dominant, VERIFIED via committed `balfinder_validation.py`
  (fig6); its output `metrics.json` is untracked (mock, public). OUTSIDE.
* **metal ~0.07%** and **Lyman-β ~0** — mechanism committed (`decompose_highn_fp.py`), but the
  magnitudes trace only to private notes → **UNVERIFIED**.
* **FP-model bracket** — report BOTH `purity_mixture` (f1784fc) and `loa0` (d496f42, headline)
  as a bracket; VERIFIED (two full artifacts, both routines committed). OUTSIDE (model choice).
* **z>4 extrapolation** — `UNVERIFIED / UNBOUNDED`, sign unknown, OUTSIDE. The mock truth caps
  at z≈3.5–3.79, so **no mock bounds the bias** in the `[4.0,4.25]` bin. This is an error term
  with no mock able to constrain it — it must **not** be presented as if the MC band covered it.

## 5 — Privacy self-check (before you commit)

Executed outputs embed private real-LOA values. The strong guarantee is simple: **a committed
notebook with zero code-cell outputs cannot leak a value.** `assert_no_outputs` enforces exactly
that; `scan_notebook_outputs` is a softer tripwire that flags decimal numbers surviving in any
output. Both read the notebook JSON — they do not execute anything.

In [ ]:
# Soft tripwire demo on a SYNTHETIC notebook (self-contained; no scratch dependency).
fake = {"cells": [
    {"cell_type": "code", "outputs": [
        {"output_type": "stream", "name": "stdout",
         "text": "SENTINEL 424242.4242  (obviously-fake demo value, not a result)"}]},
    {"cell_type": "markdown", "source": "no outputs here"},
]}
with tempfile.NamedTemporaryFile("w", suffix=".ipynb", delete=False) as tf:
    json.dump(fake, tf); tmp = tf.name
hits = privacy.scan_notebook_outputs(tmp)
print("scan on synthetic nb -> flagged outputs:", len(hits))
for h in hits:
    print(f"   cell#{h.cell_index} {h.kind}: {h.n_numbers} number(s) e.g. {h.sample!r}")
try:
    privacy.assert_no_outputs(tmp)
    print("assert_no_outputs(synthetic): PASS (BUG — it had outputs)")
except RuntimeError as e:
    print("assert_no_outputs(synthetic): correctly RAISED:", str(e)[:80])
os.remove(tmp)

print()
print(privacy.CLEAR_INSTRUCTION)

# After clearing outputs, verify THIS committed notebook is clean:
this_nb = os.path.join(REPO, "notebooks", "UNBLIND_00_guards_and_data.ipynb")
if os.path.exists(this_nb):
    n_out = len(privacy.scan_notebook_outputs(this_nb))
    print(f"\n(FYI) committed copy currently shows {n_out} output(s) with numbers "
          f"— must be 0 at commit time; run the clear command above if nonzero.)")

## Commit checklist

1. **Strip outputs** (private values live in them):
   `jupyter nbconvert --clear-output --inplace notebooks/UNBLIND_00_guards_and_data.ipynb`
2. Verify zero outputs: `assert_no_outputs('notebooks/UNBLIND_00_guards_and_data.ipynb')`.
3. Keep any executed copy out of git (e.g. `notebooks/*.exec.ipynb`, `notebooks/probe*.ipynb`
   in `.gitignore`).
4. The integrator commits — **do not commit real-LOA outputs, ever.**